# A5 Annotation Sampling & AI-Assisted Silver Annotation (CPU)

Menjalankan *annotation sampling* dan *silver annotation* (weak supervision) secara
reproduktif lewat `sipature_ml`. Ikuti `docs/annotation-runbook.md` dan
`docs/taxonomy-annotation-report.md` sebelum eksekusi.

Input: `data/processed/canonical_reviews.parquet` (dari notebook `02`).
Output: `data/annotations/*` (sampling audit, assignment, template, silver labels)
dan report + figure.

Catatan: label silver adalah *AI-assisted weak supervision*, bukan gold label manusia.


In [1]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
# ============================================================
# CONFIGURATION CELL — satu-satunya tempat mengubah parameter.
# ============================================================
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/SIPATURE")
DRIVE_PROCESSED_DIR = DRIVE_ROOT / "data" / "processed"

PROJECT_DIR = Path("/content/hackathon/ml")
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
ANNOTATION_DIR = PROJECT_DIR / "data" / "annotations"
REPORT_DIR = PROJECT_DIR / "artifacts" / "reports"
FIGURE_DIR = PROJECT_DIR / "artifacts" / "figures" / "annotation"

DRIVE_ANNOTATION_DIR = DRIVE_ROOT / "data" / "annotations"
DRIVE_REPORT_DIR = DRIVE_ROOT / "reports"
DRIVE_FIGURE_DIR = DRIVE_ROOT / "figures" / "annotation"

print("Drive root:", DRIVE_ROOT)
print("Sumber canonical reviews:", DRIVE_PROCESSED_DIR / "canonical_reviews.parquet")
print("Annotation dir :", ANNOTATION_DIR)
print("Processed dir  :", PROCESSED_DIR)


Drive root: /content/drive/MyDrive/SIPATURE
Sumber canonical reviews: /content/drive/MyDrive/SIPATURE/data/processed/canonical_reviews.parquet
Annotation dir : /content/hackathon/ml/data/annotations
Processed dir  : /content/hackathon/ml/data/processed


In [4]:
from google.colab import userdata
import base64
import os
import shutil
import subprocess

token = userdata.get("GITHUB_TOKEN")
assert token, "GITHUB_TOKEN tidak ditemukan di Colab Secrets"

credentials = f"x-access-token:{token}"
authorization = base64.b64encode(credentials.encode()).decode()

repo_dir = "/content/hackathon"
shutil.rmtree(repo_dir, ignore_errors=True)

environment = os.environ.copy()
environment["GIT_CONFIG_COUNT"] = "1"
environment["GIT_CONFIG_KEY_0"] = "http.extraHeader"
environment["GIT_CONFIG_VALUE_0"] = f"Authorization: Basic {authorization}"

result = subprocess.run(
    ["git", "clone", "https://github.com/jodypangaribuan/hackathon.git", repo_dir],
    env=environment,
    text=True,
    capture_output=True,
)

print("Return code:", result.returncode)
print(result.stdout)
print(result.stderr)

assert result.returncode == 0, "Clone gagal. Periksa izin token GitHub."


Return code: 0

Cloning into '/content/hackathon'...



In [5]:
%cd /content/hackathon/ml
!git log --oneline -3


/content/hackathon/ml
3a4d352 (HEAD -> main, origin/main, origin/HEAD) feat: add annotation sampling and silver annotation notebook
ac8b269 docs: record notebook 01-02 completion and mark EDA/cleaning artifacts done
06570f5 Created using Colab


In [6]:
%cd /content/hackathon/ml
!python -m pip install -r requirements-colab.lock.txt
!python -m pip install --no-deps -e .


/content/hackathon/ml
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 4.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 79.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 95.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.1/42.1 MB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 99.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.3/37.3 MB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 80.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 88.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 821.0/821.0 MB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [3]:
import numpy
import pandas
import pyarrow
import sklearn
import matplotlib

print("NumPy:", numpy.__version__)
print("Pandas:", pandas.__version__)
print("PyArrow:", pyarrow.__version__)
print("Scikit-learn:", sklearn.__version__)
print("Matplotlib:", matplotlib.__version__)


NumPy: 2.2.6
Pandas: 2.3.3
PyArrow: 19.0.1
Scikit-learn: 1.7.2
Matplotlib: 3.10.3


In [4]:
# Salin canonical_reviews.parquet (output notebook 02) dari Drive ke lokal.
import shutil
from pathlib import Path

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

source = DRIVE_PROCESSED_DIR / "canonical_reviews.parquet"
assert source.is_file(), (
    f"canonical_reviews.parquet tidak ditemukan di Drive: {source}\n"
    "Jalankan notebook 02 terlebih dahulu dan pastikan output sudah disalin ke Drive."
)

destination = PROCESSED_DIR / "canonical_reviews.parquet"
shutil.copy2(source, destination)
print("Disalin:", source.name, "->", destination)


Disalin: canonical_reviews.parquet -> /content/hackathon/ml/data/processed/canonical_reviews.parquet


In [5]:
import sys
from pathlib import Path

source_dir = PROJECT_DIR / "src"

assert source_dir.is_dir(), "Folder source SIPATURE tidak ditemukan."

if str(source_dir) not in sys.path:
    sys.path.insert(0, str(source_dir))

import sipature_ml

print("Modul SIPATURE berhasil dimuat dari:")
print(sipature_ml.__file__)


Modul SIPATURE berhasil dimuat dari:
/content/hackathon/ml/src/sipature_ml/__init__.py


In [6]:
from sipature_ml.config import load_config, ML_ROOT

taxonomy = load_config("taxonomy")
sampling = taxonomy["sampling"]

taxonomy_path = ML_ROOT / "configs" / "taxonomy.yaml"
schema_path = ML_ROOT / "contracts" / "annotation.schema.json"
assert taxonomy_path.is_file() and schema_path.is_file(), "taxonomy/schema tidak ditemukan"

print("Taxonomy version:", taxonomy["taxonomy_version"])
print("Status:", taxonomy["status"])
print("Jumlah aspek:", len(taxonomy["aspect_definitions"]))
print("Pilot size:", sampling["pilot_size"])
print("Main size :", sampling["main_size"])
print("Double annotation rate:", sampling["double_annotation_rate"])
print("Seed:", sampling["seed"])


Taxonomy version: 1.0.0-rc1
Status: pilot_locked
Jumlah aspek: 14
Pilot size: 120
Main size : 1200
Double annotation rate: 0.2
Seed: 42


In [7]:
from sipature_ml.annotation import run_annotation_sampling

sampling_summary = run_annotation_sampling(PROCESSED_DIR, ANNOTATION_DIR, REPORT_DIR)

print("Annotation version :", sampling_summary["annotation_version"])
print("Clean text pool    :", sampling_summary["clean_text_pool"])
print("Pilot unique reviews:", sampling_summary["pilot_unique_reviews"])
print("Main unique reviews :", sampling_summary["main_unique_reviews"])
print("Main annotation tasks:", sampling_summary["main_annotation_tasks"])
print("Main double annotated:", sampling_summary["main_double_annotated_reviews"])
print("Sample overlap (harus 0):", sampling_summary["sample_overlap"])
print("Destinations in main:", sampling_summary["destinations_in_main"])
print("Assignment files:")
for name in sampling_summary["assignment_files"]:
    print("  -", name)


Annotation version : 1.0.0-rc1
Clean text pool    : 12234
Pilot unique reviews: 120
Main unique reviews : 1200
Main annotation tasks: 1440
Main double annotated: 240
Sample overlap (harus 0): 0
Destinations in main: 262
Assignment files:
  - main_A1_annotations.jsonl
  - main_A2_annotations.jsonl
  - main_A3_annotations.jsonl
  - pilot_A1_annotations.jsonl
  - pilot_A2_annotations.jsonl
  - pilot_A3_annotations.jsonl


In [8]:
from sipature_ml.annotation import run_silver_annotation

silver_summary = run_silver_annotation(PROCESSED_DIR, ANNOTATION_DIR, REPORT_DIR)

print("Silver version :", silver_summary["silver_version"])
print("Total records  :", silver_summary["total_records"])
print("Status counts  :", silver_summary["status_counts"])
print("Mean pass agreement:", silver_summary["mean_pass_agreement"])
print("Disagreement queue records:", silver_summary["disagreement_queue_records"])
print("Main aspect support:", silver_summary["main_aspect_support"])
print("Main polarity support:", silver_summary["main_polarity_support"])
print("Main severity support:", silver_summary["main_severity_support"])


Silver version : silver-1.0.0
Total records  : 1320
Status counts  : {'consensus': 489, 'no_supported_aspect': 497, 'review_recommended': 334}
Mean pass agreement: 0.8827
Disagreement queue records: 334
Main aspect support: {'access': 112, 'cleanliness': 166, 'comfort': 135, 'crowding': 36, 'maintenance': 70, 'opening_hours': 10, 'parking': 133, 'price_transparency': 181, 'public_facilities': 80, 'safety': 30, 'sanitation': 118, 'scenery': 203, 'staff_service': 187, 'waste': 42}
Main polarity support: {'negative': 477, 'neutral': 368, 'positive': 658}
Main severity support: {'high': 25, 'low': 253, 'medium': 199}


In [9]:
from sipature_ml.quality_figures import (
    generate_annotation_figures,
    generate_silver_figures,
)

annotation_figures = generate_annotation_figures(ANNOTATION_DIR, REPORT_DIR, FIGURE_DIR)
silver_figures = generate_silver_figures(ANNOTATION_DIR, FIGURE_DIR)

print("Annotation figures:", len(annotation_figures))
for name in annotation_figures:
    print("-", name)
print("Silver figures:", len(silver_figures))
for name in silver_figures:
    print("-", name)


Annotation figures: 6
- 23_annotation_candidate_support.png
- 24_pilot_sampling_stratification.png
- 25_main_annotation_assignments.png
- 26_taxonomy_group_composition.png
- 27_candidate_aspect_cooccurrence.png
- 28_main_destination_coverage.png
Silver figures: 5
- 29_silver_aspect_distribution.png
- 30_silver_polarity_severity.png
- 31_silver_aspect_cooccurrence.png
- 32_silver_status_distribution.png
- 33_silver_pass_agreement.png


In [10]:
# Salin output annotations + report + figure ke Drive (artefak persisten).
import shutil
from pathlib import Path

for local_dir, drive_dir in (
    (ANNOTATION_DIR, DRIVE_ANNOTATION_DIR),
    (REPORT_DIR, DRIVE_REPORT_DIR),
    (FIGURE_DIR, DRIVE_FIGURE_DIR),
):
    drive_dir.mkdir(parents=True, exist_ok=True)
    for source in sorted(local_dir.glob("*")):
        if source.is_file():
            shutil.copy2(source, drive_dir / source.name)
            print(f"Disalin: {source.name} -> {drive_dir}")


Disalin: README.md -> /content/drive/MyDrive/SIPATURE/data/annotations
Disalin: annotation_assignments.csv -> /content/drive/MyDrive/SIPATURE/data/annotations
Disalin: main_A1_annotations.jsonl -> /content/drive/MyDrive/SIPATURE/data/annotations
Disalin: main_A2_annotations.jsonl -> /content/drive/MyDrive/SIPATURE/data/annotations
Disalin: main_A3_annotations.jsonl -> /content/drive/MyDrive/SIPATURE/data/annotations
Disalin: main_sampling_audit.csv -> /content/drive/MyDrive/SIPATURE/data/annotations
Disalin: pilot_A1_annotations.jsonl -> /content/drive/MyDrive/SIPATURE/data/annotations
Disalin: pilot_A2_annotations.jsonl -> /content/drive/MyDrive/SIPATURE/data/annotations
Disalin: pilot_A3_annotations.jsonl -> /content/drive/MyDrive/SIPATURE/data/annotations
Disalin: pilot_sampling_audit.csv -> /content/drive/MyDrive/SIPATURE/data/annotations
Disalin: silver-disagreement-queue.jsonl -> /content/drive/MyDrive/SIPATURE/data/annotations
Disalin: silver-pass-balanced.jsonl -> /content/driv

In [11]:
# ============================================================
# RUN SUMMARY — output path, hash, dan limitations.
# ============================================================
from sipature_ml.manifest import sha256_file
from sipature_ml.config import ML_ROOT

print("TAXONOMY VERSION:", sampling_summary["annotation_version"])
print("SILVER VERSION  :", silver_summary["silver_version"])

print("\nTAXONOMY SHA256 :", sampling_summary["taxonomy_sha256"])
print("SCHEMA SHA256    :", sampling_summary["schema_sha256"])
print("SILVER SHA256    :", silver_summary["silver_sha256"])

print("\nOUTPUT ANNOTATION DIR :", ANNOTATION_DIR)
print("OUTPUT REPORT DIR     :", REPORT_DIR)
print("OUTPUT FIGURE DIR     :", FIGURE_DIR)
print("DRIVE ANNOTATION DIR  :", DRIVE_ANNOTATION_DIR)

print("\nLIMITATIONS:")
for item in silver_summary["limitations"]:
    print("  -", item)


TAXONOMY VERSION: 1.0.0-rc1
SILVER VERSION  : silver-1.0.0

TAXONOMY SHA256 : 9840978b6c62613cb580ba8c736558fda12b688ae38cd248e01428daacd727f5
SCHEMA SHA256    : ba0cac8842e7efbf48d4a883d2aae6895c68840d26ef181fb44a72e9719950ec
SILVER SHA256    : 8838930b046def5303c89efb4f018d9a5d8a77cc2b142fa25d4c445f4d9d2610

OUTPUT ANNOTATION DIR : /content/hackathon/ml/data/annotations
OUTPUT REPORT DIR     : /content/hackathon/ml/artifacts/reports
OUTPUT FIGURE DIR     : /content/hackathon/ml/artifacts/figures/annotation
DRIVE ANNOTATION DIR  : /content/drive/MyDrive/SIPATURE/data/annotations

LIMITATIONS:
  - Silver labels are AI-assisted weak-supervision outputs, not human gold labels.
  - Pass agreement measures rule consistency, not inter-annotator agreement.
  - Confidence is vote agreement and is not calibrated probability.
  - Seed lexicons may miss implicit, sarcastic, or domain-specific statements.
